## STEP 01: Generate our Dataset

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_customers = 10000

cities = [
    "Kinshasa",
    "Lubumbashi",
    "Goma",
    "Matadi",
    "Mbuji-Mayi",
    "Kananga",
    "Kisangani",
    "Kolwezi",
    "Bukavu",
    "Likasi"
]

customers = pd.DataFrame({
    "customer_id":[f"C{i:05d}" for i in range(1,n_customers+1)],
    "age":np.random.randint(18,70,n_customers),
    "gender":np.random.choice(["Male","Female"],n_customers),
    "city":np.random.choice(cities,n_customers),
})

In [2]:
customers.head(10)

,customer_id,age,gender,city
0,C00001,56,Male,Kinshasa
1,C00002,69,Female,Kananga
2,C00003,46,Female,Matadi
3,C00004,32,Female,Mbuji-Mayi
4,C00005,60,Male,Mbuji-Mayi
5,C00006,25,Female,Lubumbashi
6,C00007,38,Female,Kolwezi
7,C00008,56,Female,Lubumbashi
8,C00009,36,Male,Matadi
9,C00010,40,Male,Mbuji-Mayi


In [3]:
customers.shape

(10000, 4)

## Ajouter les offres télécom

In [4]:
plans = pd.DataFrame({
    "plan_id":[1,2,3,4],
    "plan_name":[
        "Basic",
        "Silver",
        "Gold",
        "Unlimited"
    ],
    "monthly_fee":[5,10,20,40]
})

## Attribuer une offre à chaque client

In [5]:
customers["plan_id"] = np.random.choice(
    plans["plan_id"],
    n_customers,
    p=[0.4,0.3,0.2,0.1]
)

## Save

In [6]:
customers.to_csv("../data/customers.csv",index=False)
plans.to_csv("../data/plans.csv",index=False)

In [11]:
customers.head()

,customer_id,age,gender,city,plan_id
0,C00001,56,Male,Kinshasa,3
1,C00002,69,Female,Kananga,3
2,C00003,46,Female,Matadi,4
3,C00004,32,Female,Mbuji-Mayi,1
4,C00005,60,Male,Mbuji-Mayi,2


In [12]:
customers.shape

(10000, 5)

## STEP 02: Generating usage data

In [13]:
months = pd.date_range(
    start="2024-01-01",
    end="2025-12-01",
    freq="MS"
)

len(months)

24

## Create usage data

In [14]:
usage_data = []

for _, customer in customers.iterrows():

    for month in months:

        plan = customer["plan_id"]

        if plan == 1:
            data_gb = np.random.normal(5,2)
            calls = np.random.normal(100,20)
            sms = np.random.normal(50,10)

        elif plan == 2:
            data_gb = np.random.normal(15,5)
            calls = np.random.normal(250,50)
            sms = np.random.normal(100,20)

        elif plan == 3:
            data_gb = np.random.normal(35,10)
            calls = np.random.normal(500,80)
            sms = np.random.normal(150,30)

        else:
            data_gb = np.random.normal(80,15)
            calls = np.random.normal(1000,100)
            sms = np.random.normal(200,40)

        usage_data.append([
            customer["customer_id"],
            month,
            round(max(0,data_gb),2),
            round(max(0,calls),0),
            round(max(0,sms),0)
        ])

## Creating the DataFrame

In [15]:
usage = pd.DataFrame(
    usage_data,
    columns=[
        "customer_id",
        "month",
        "data_gb",
        "call_minutes",
        "sms"
    ]
)

In [16]:
usage.head()

,customer_id,month,data_gb,call_minutes,sms
0,C00001,2024-01-01,23.28,611.0,222.0
1,C00001,2024-02-01,19.49,453.0,168.0
2,C00001,2024-03-01,25.07,478.0,155.0
3,C00001,2024-04-01,27.87,507.0,178.0
4,C00001,2024-05-01,34.22,417.0,185.0


In [17]:
usage.shape

(240000, 5)

## Save

In [18]:
usage.to_csv("../data/usage.csv", index=False)

## STEP 03: Add the most important dimension → REVENUE 💰

## Generating top-ups

In [19]:
recharges = []

for _, row in usage.iterrows():

    base_price = {
        1: 5,
        2: 10,
        3: 20,
        4: 40
    }

    plan_price = base_price[customers.loc[
        customers["customer_id"] == row["customer_id"], "plan_id"
    ].values[0]]

    # Simulate additional income (overage / bonus)
    extra = np.random.normal(2, 1)

    amount = max(0, plan_price + extra)

    recharges.append([
        row["customer_id"],
        row["month"],
        round(amount, 2)
    ])

## DataFrame revenue

In [20]:
revenue = pd.DataFrame(
    recharges,
    columns=["customer_id", "month", "revenue"]
)

In [21]:
revenue.head()

,customer_id,month,revenue
0,C00001,2024-01-01,22.12
1,C00001,2024-02-01,21.36
2,C00001,2024-03-01,21.56
3,C00001,2024-04-01,20.70
4,C00001,2024-05-01,21.30


In [22]:
revenue.shape

(240000, 3)

In [23]:
revenue.to_csv("../data/revenue.csv", index=False)